# Detecção de Spam com Transformers

## Objetivo

Avaliar uma abordagem baseada em Transformer para identificar
mensagens spam e ham, comparando os resultados com os modelos
treinados com TF-IDF e BoW + n-gram.

## Estratégia de comparação

- Utilizar a mesma base e divisão de treino e teste.
- Preservar o texto original para o tokenizador do modelo.
- Comparar precisão, recall e F1 da classe spam.
- Analisar falsos positivos e falsos negativos.

## 1. Preparação do ambiente

Bibliotecas utilizadas:

- **Pandas:** manipulação dos dados.
- **PyTorch:** execução do Transformer.
- **Transformers:** carregamento do modelo e do tokenizador.
- **Scikit-learn:** divisão dos dados e métricas de avaliação.
- **Matplotlib:** visualização da matriz de confusão.
- **time:** medição do tempo de execução.

In [29]:
# Dados e visualização
import pandas as pd
import matplotlib.pyplot as plt

# Medição de tempo
from time import perf_counter

# Transformers
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed
from transformers import AutoModelForSequenceClassification

# Divisão dos dados
from sklearn.model_selection import train_test_split

# Avaliação
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

print("Bibliotecas Importadas com Sucesso!!")

Bibliotecas Importadas com Sucesso!!


In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

set_seed(RANDOM_STATE)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo:", device)

Dispositivo: cpu


# 2. Importando a Base de Dados

In [3]:
dados = pd.read_csv("../data/dados_spam.csv", index_col=0)
dados.head()

,categoria,texto
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
print("Quantidade de mensagens:", len(dados))

display(dados[["texto", "categoria"]].head())

print("\nValores ausentes:")
print(dados[["texto", "categoria"]].isna().sum())

print("\nDistribuição das categorias:")
print(dados["categoria"].value_counts())

Quantidade de mensagens: 5574


,texto,categoria
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives aro...",ham



Valores ausentes:
texto        0
categoria    0
dtype: int64

Distribuição das categorias:
categoria
ham     4827
spam     747
Name: count, dtype: int64


## 3. Divisão de treino e teste

Usaremos 80% dos exemplos para treino e 20% para teste.

A estratificação preserva aproximadamente a proporção
de spam e ham em cada conjunto.

O Transformer receberá o texto original. A preparação
numérica será realizada pelo tokenizador do modelo.

In [5]:
X = dados["texto"]
y = dados["categoria"]

In [6]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

print("Total treino:", len(X_treino))
print("Total teste:", len(X_teste))

Total treino: 4459
Total teste: 1115


## 4. Carregamento do modelo pré-treinado

Usaremos o BERT-Tiny já ajustado para detectar spam
em mensagens em inglês.

O `pipeline` realiza automaticamente:
- Tokenização do texto.
- Execução do modelo.
- Retorno da categoria prevista e sua pontuação.

Não realizaremos treinamento adicional.

Nota: esse modelo foi ajustado no dataset sms_spam.
Possível sobreposição com nossa base limita a interpretação
dos resultados como avaliação independente.

In [35]:
from transformers import pipeline

classificador = pipeline(
    task="text-classification",
    model="mrm8488/bert-tiny-finetuned-sms-spam-detection",
    device=-1
)

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

c:\Users\Unisalesiano\Documents\Renan\Projetos Renan\sms-spam-detection\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Unisalesiano\.cache\huggingface\hub\models--mrm8488--bert-tiny-finetuned-sms-spam-detection. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 17.6MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/324 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## 5. Teste inicial de inferência

Vamos enviar duas mensagens para conferir o funcionamento.

O resultado apresenta:
- `label`: categoria prevista pelo modelo.
- `score`: pontuação atribuída à categoria prevista.

O score não representa a acurácia do modelo.

In [36]:
mensagens = [
    "Hi, I will call you after the meeting.",
    "Congratulations! You won a cash prize. Call now to claim!"
]

resultados = classificador(
    mensagens,
    truncation=True,
    max_length=512
)

for mensagem, resultado in zip(mensagens, resultados):
    print("Mensagem:", mensagem)
    print("Resultado:", resultado)
    print()

Mensagem: Hi, I will call you after the meeting.
Resultado: {'label': 'LABEL_0', 'score': 0.9384437799453735}

Mensagem: Congratulations! You won a cash prize. Call now to claim!
Resultado: {'label': 'LABEL_0', 'score': 0.9356968402862549}



In [37]:
texto = (
    "Camera - You are awarded a SiPix Digital Camera! "
    "call 09061221066 from landline. Delivery within 28 days."
)

classificador(texto, truncation=True, max_length=512)

[{'label': 'LABEL_1', 'score': 0.9023539423942566}]

### Verificação inicial das previsões

Vamos aplicar o modelo a uma pequena amostra balanceada, usando os textos
originais, e comparar as previsões com as categorias reais.

Essa etapa verifica o funcionamento; não representa a avaliação final.

In [38]:
# Selecionar 5 mensagens de cada categoria
amostra = pd.concat([
    dados[dados["categoria"] == "ham"].sample(5, random_state=42),
    dados[dados["categoria"] == "spam"].sample(5, random_state=42)
]).copy()

# Classificar os textos originais
resultados = classificador(
    amostra["texto"].tolist(),
    truncation=True,
    max_length=512
)

# Traduzir os rótulos do modelo
rotulos = {"LABEL_0": "ham", "LABEL_1": "spam"}

amostra["previsao"] = [rotulos[r["label"]] for r in resultados]
amostra["acertou"] = amostra["categoria"] == amostra["previsao"]

amostra[["texto", "categoria", "previsao", "acertou"]]

,texto,categoria,previsao,acertou
688,"Dear,Me at cherthala.in case u r coming cochin...",ham,ham,True
2520,Ok. I only ask abt e movie. U wan ktv oso?,ham,ham,True
1079,Convey my regards to him,ham,ham,True
1724,"Hi Jon, Pete here, Ive bin 2 Spain recently & ...",ham,spam,False
1974,I had askd u a question some hours before. Its...,ham,ham,True
1456,Summers finally here! Fancy a chat or flirt wi...,spam,spam,True
1853,This is the 2nd time we have tried 2 contact u...,spam,spam,True
673,Get ur 1st RINGTONE FREE NOW! Reply to this ms...,spam,spam,True
947,Ur cash-balance is currently 500 pounds - to m...,spam,spam,True
2881,Last Chance! Claim ur £150 worth of discount v...,spam,spam,True
